In [4]:
import pandas as pd
import numpy as np

# Load datasets
symbols = pd.read_csv("data/symbols.csv", sep=";")

transactions = pd.read_csv(
    "data/account-statement-1-1-2024-12-31-2024.csv",
    sep=";"
)

countries = pd.read_csv("data/country.csv")



# Dataset dimensions
print("=== Dataset Shapes ===")
print(f"Symbols: {symbols.shape}")
print(f"Transactions: {transactions.shape}")
print(f"Countries: {countries.shape}")

# Preview datasets
print("\n=== Symbols Preview ===")
print(symbols.head())

print("\n=== Transactions Preview ===")
print(transactions.head())

print("\n=== Countries Preview ===")
print(countries.head())

=== Dataset Shapes ===
Symbols: (3194, 5)
Transactions: (2745, 6)
Countries: (249, 11)

=== Symbols Preview ===
  symbol                   company_name              sector  \
0   TEAM          Atlassian Corporation          Technology   
1    WDS  Woodside Energy Group Limited              Energy   
2    OSW   OneSpaWorld Holdings Limited   Consumer Cyclical   
3   ACGL        Arch Capital Group Ltd.  Financial Services   
4    AGO          Assured Guaranty Ltd.  Financial Services   

                  industry    country  
0   Software - Application  Australia  
1            Oil & Gas E&P  Australia  
2                  Leisure    Bahamas  
3  Insurance - Diversified    Bermuda  
4    Insurance - Specialty    Bermuda  

=== Transactions Preview ===
   IDTransaction                 Date TransactionType Symbol    Unit  \
0   2.769834e+09  11/01/2024 10:44:03             BUY    BAP  1605.0   
1   2.767325e+09  24/01/2024 08:07:24            SELL    BAP  1605.0   
2   2.815474e+09  10/01

In [5]:
import os
print(os.getcwd())

c:\Users\UserONE\Desktop\robaBDA2026\homew3


In [6]:
#2.1
# 1. Check for missing values in the transactions dataset
print("=== Missing Values Before Cleaning ===")
print("Transactions Nulls:\n", transactions.isnull().sum(), "\n")

# 2. Remove rows with missing values in critical transaction fields
transactions = transactions.dropna(subset=[ 'Date', 'TransactionType', 'Symbol', 'Unit'])

# 3. Keep only the attributes required by the Star Schema model
# This automatically removes any unused columns such as 'Unnamed: 5'
required_transaction_cols = ['Date', 'TransactionType', 'Symbol', 'Unit']
transactions = transactions[required_transaction_cols]

print("=== Missing Values After Cleaning and Column Drop ===")
print("Transactions Nulls:\n", transactions.isnull().sum(), "\n")
print("Remaining Columns in Transactions:", transactions.columns.tolist())

=== Missing Values Before Cleaning ===
Transactions Nulls:
 IDTransaction       464
Date                464
TransactionType     464
Symbol              464
Unit                464
Unnamed: 5         2745
dtype: int64 

=== Missing Values After Cleaning and Column Drop ===
Transactions Nulls:
 Date               0
TransactionType    0
Symbol             0
Unit               0
dtype: int64 

Remaining Columns in Transactions: ['Date', 'TransactionType', 'Symbol', 'Unit']


In [7]:
# 3. Verify that every transaction symbol exists in the symbols dataset
unmapped_symbols = transactions[~transactions['Symbol'].isin(symbols['symbol'])]['Symbol'].unique()
print("=== Referential Integrity Checks ===")
print(f"Number of unmapped symbols: {len(unmapped_symbols)}")

# 4. Verify that every company country can be mapped to the country dataset
symbols['country'] = symbols['country'].str.strip()
countries['name'] = countries['name'].str.strip()

unmapped_countries = symbols[~symbols['country'].isin(countries['name'])]['country'].unique()
print(f"Number of unmapped countries: {len(unmapped_countries)}\n")

=== Referential Integrity Checks ===
Number of unmapped symbols: 18
Number of unmapped countries: 2



In [9]:
# Create Dim_TransactionType
dim_transaction_type = pd.DataFrame({
    'transaction_type': transactions['TransactionType'].unique()
}).reset_index(drop=True)

dim_transaction_type.insert(0, 'type_key', dim_transaction_type.index + 1)


# Create Dim_Geography
dim_geography = countries[['alpha-2', 'name', 'region', 'sub-region']].copy()

dim_geography.columns = ['country_code', 'country', 'region', 'sub_region']
dim_geography = dim_geography.drop_duplicates().reset_index(drop=True)

dim_geography.insert(0, 'geo_key', dim_geography.index + 1)


# Create Dim_Symbol
dim_symbol = symbols[['symbol', 'company_name', 'sector', 'industry']].copy()

dim_symbol.columns = ['symbol', 'company', 'sector', 'industry']
dim_symbol = dim_symbol.drop_duplicates().reset_index(drop=True)

dim_symbol.insert(0, 'symbol_key', dim_symbol.index + 1)


# Create Dim_Time (optimized - no repeated conversions)
transactions['Parsed_Date'] = pd.to_datetime(transactions['Date'], errors='coerce')

dim_time = pd.DataFrame({
    'date': transactions['Parsed_Date'].dt.date.unique()
}).sort_values('date').reset_index(drop=True)

dim_time.insert(0, 'time_key', dim_time.index + 1)

# compute datetime once only
dim_time['date'] = pd.to_datetime(dim_time['date'])

dim_time['day'] = dim_time['date'].dt.day
dim_time['month'] = dim_time['date'].dt.month
dim_time['year'] = dim_time['date'].dt.year


dim_time = dim_time.dropna(subset=['year'])
dim_time['year'] = dim_time['year'].astype(int)


dim_time['quarter'] = 'Q' + dim_time['date'].dt.quarter.fillna(0).astype(int).astype(str)

# Remove bad values
dim_time = dim_time[dim_time['quarter'] != 'Q0']

dim_time['day_of_week'] = dim_time['date'].dt.day_name()

# Safety cleaning (important for Q4 issue you had)
dim_time = dim_time.dropna(subset=['quarter'])

print("=== Dimensions Status ===")
print("All dimension dataframes created successfully with Surrogate Keys.")

=== Dimensions Status ===
All dimension dataframes created successfully with Surrogate Keys.


In [10]:
# 1. Strip time completely and normalize both date columns to pure date formats (Crucial Fix!)
transactions['date_only'] = pd.to_datetime(pd.to_datetime(transactions['Date'], errors='coerce').dt.date)
dim_time['date'] = pd.to_datetime(pd.to_datetime(dim_time['date']).dt.date)

# 2. Mapping to Dim_Time (Now types and values match perfectly)
fact_transactions = transactions.merge(
    dim_time,
    left_on='date_only',
    right_on='date',
    how='inner'
)

# 3. Mapping to Dim_Symbol
fact_transactions = fact_transactions.merge(
    dim_symbol,
    left_on='Symbol',
    right_on='symbol',
    how='inner'
)

# 4. Mapping to Dim_TransactionType
fact_transactions = fact_transactions.merge(
    dim_transaction_type,
    left_on='TransactionType',
    right_on='transaction_type',
    how='inner'
)

# 5. Add country mapping (from original symbols dataset → geography)
fact_transactions = fact_transactions.merge(
    symbols[['symbol', 'country']],
    left_on='Symbol',
    right_on='symbol',
    how='inner'
)

fact_transactions = fact_transactions.merge(
    dim_geography,
    on='country',
    how='inner'
)

# 6. Final Fact Table (Keep ONLY measures + surrogate keys)
fact_transactions['quantity'] = fact_transactions['Unit']

fact_transactions = fact_transactions[[
    'time_key',
    'symbol_key',
    'geo_key',
    'type_key',
    'quantity'
]]

print("=== Final Fact Table Preview ===")
print(f"Total Rows in Fact Table: {fact_transactions.shape[0]}")
print(fact_transactions.head())

=== Final Fact Table Preview ===
Total Rows in Fact Table: 842
   time_key  symbol_key  geo_key  type_key  quantity
0        77         284      175         1    1605.0
1        71         284      175         2     914.0
2        34         258      131         2     515.0
3        77           4       25         1     320.0
4        72         342      235         2     309.0


In [11]:
#2.2
print("=== Question 2 (Fixed Version) ===")

# 1. Get the surrogate key for 'BUY' transactions
buy_type_keys = dim_transaction_type[dim_transaction_type['transaction_type'] == 'BUY']['type_key']

# 2. Advanced filtering for Q4 and Year 2024 to handle float/string formats safely
# تحويل العمود إلى نصوص أولاً ثم الفحص لتجنب مشاكل الكسور
dim_time['quarter_str'] = dim_time['quarter'].astype(str)
dim_time['year_str'] = dim_time['year'].astype(str)

q4_2024_time_keys = dim_time[
    dim_time['quarter_str'].str.contains('Q4', na=False) & 
    dim_time['year_str'].str.contains('2024', na=False)
]['time_key']

# 3. Filter the Fact Table using the keys found
q2_fact = fact_transactions[
    fact_transactions['type_key'].isin(buy_type_keys) & 
    fact_transactions['time_key'].isin(q4_2024_time_keys)
]

# 4. Merge with Dim_Symbol to group by Industry and show results
if q2_fact.shape[0] > 0:
    q2_merged = q2_fact.merge(dim_symbol, on='symbol_key')
    q2_result = q2_merged.groupby('industry').size().reset_index(name='number_of_transactions')
    q2_result = q2_result.sort_values(by='number_of_transactions', ascending=False).head(5)
    print(q2_result.to_string(index=False))
else:
    print("🚨 Alert: Fact table subset is still empty for Q4 2024 BUY transactions.")
    print(f"Current rows in fact_transactions: {fact_transactions.shape[0]}")
    print(f"Matched BUY keys: {len(buy_type_keys)}, Matched Q4 2024 keys: {len(q4_2024_time_keys)}")

=== Question 2 (Fixed Version) ===
                      industry  number_of_transactions
                Semiconductors                      16
               Credit Services                       7
     Software - Infrastructure                       7
Internet Content & Information                       6
              Telecom Services                       6


In [12]:
print("\n=== Question 5 ===")
# 1. Filter dimensions for 'BUY' and Year 2024.0
buy_type_keys = dim_transaction_type[dim_transaction_type['transaction_type'] == 'BUY']['type_key']
year_2024_time_keys = dim_time[dim_time['year'] == 2024.0]['time_key']

# 2. Filter Fact Table
q5_fact = fact_transactions[
    fact_transactions['type_key'].isin(buy_type_keys) & 
    fact_transactions['time_key'].isin(year_2024_time_keys)
]

# 3. Merge with Dim_Geography and sum the metric 'quantity'
q5_merged = q5_fact.merge(dim_geography, on='geo_key')
q5_result = q5_merged.groupby('region')['quantity'].sum().reset_index(name='total_units_bought')
q5_result = q5_result.sort_values(by='total_units_bought', ascending=False).head(5)

print(q5_result.to_string(index=False))


=== Question 5 ===
  region  total_units_bought
Americas             16197.0
  Europe              7793.0
    Asia              4013.0


In [13]:
print("\n=== Question 9 ===")
# 1. Filter dimensions for 'Asia' region and Year 2024.0
asia_geo_keys = dim_geography[dim_geography['region'] == 'Asia']['geo_key']
year_2024_time_keys = dim_time[dim_time['year'] == 2024.0]['time_key']

# 2. Filter Fact Table
q9_fact = fact_transactions[
    fact_transactions['geo_key'].isin(asia_geo_keys) & 
    fact_transactions['time_key'].isin(year_2024_time_keys)
]

# 3. Merge with Dim_Symbol to group by Industry and sum the traded quantity
q9_merged = q9_fact.merge(dim_symbol, on='symbol_key')
q9_result = q9_merged.groupby('industry')['quantity'].sum().reset_index(name='total_units_traded')
q9_result = q9_result.sort_values(by='total_units_traded', ascending=False).head(5)

print(q9_result.to_string(index=False))


=== Question 9 ===
                      industry  total_units_traded
            Auto Manufacturers              1869.0
        Software - Application               951.0
               Credit Services               932.0
           Aerospace & Defense               887.0
Internet Content & Information               827.0


In [14]:
print("\n=== Question 12 ===")
# 1. Filter dimensions for 'SELL', 'Monday', and Year 2024.0
sell_type_keys = dim_transaction_type[dim_transaction_type['transaction_type'] == 'SELL']['type_key']
monday_2024_time_keys = dim_time[(dim_time['day_of_week'] == 'Monday') & (dim_time['year'] == 2024.0)]['time_key']

# 2. Filter Fact Table
q12_fact = fact_transactions[
    fact_transactions['type_key'].isin(sell_type_keys) & 
    fact_transactions['time_key'].isin(monday_2024_time_keys)
]

# 3. Merge with Dim_Symbol to group by Sector and sum the units sold
q12_merged = q12_fact.merge(dim_symbol, on='symbol_key')
q12_result = q12_merged.groupby('sector')['quantity'].sum().reset_index(name='total_units_sold_monday')
q12_result = q12_result.sort_values(by='total_units_sold_monday', ascending=False).head(3)

print(q12_result.to_string(index=False))


=== Question 12 ===
                sector  total_units_sold_monday
            Technology                    975.0
Communication Services                    659.0
     Consumer Cyclical                    506.0


In [15]:
print("\n=== Question 13 ===")
# 1. Merge Fact Table with Geography and Symbol dimensions
q13_merged = fact_transactions.merge(dim_geography, on='geo_key')
q13_merged = q13_merged.merge(dim_symbol, on='symbol_key')

# 2. Group by Region and count DISTINCT (nunique) industries
q13_result = q13_merged.groupby('region')['industry'].nunique().reset_index(name='distinct_industries_count')
q13_result = q13_result.sort_values(by='distinct_industries_count', ascending=False).head(5)

print(q13_result.to_string(index=False))


=== Question 13 ===
  region  distinct_industries_count
Americas                         36
  Europe                         25
    Asia                          9


In [21]:
fact_transactions.to_csv("data/fact_transactions.csv", index=False)
dim_time.to_csv("data/dim_time.csv", index=False)
dim_symbol.to_csv("data/dim_symbol.csv", index=False)
dim_transaction_type.to_csv("data/dim_transaction_type.csv", index=False)

In [23]:
!python -m pip install streamlit

   ---------------------------------------- 0.0/9.2 MB ? eta -:--:--
   --------- ------------------------------ 2.1/9.2 MB 14.3 MB/s eta 0:00:01
   ---------------------------- ----------- 6.6/9.2 MB 19.0 MB/s eta 0:00:01
   ---------------------------------------- 9.2/9.2 MB 19.6 MB/s  0:00:00
   ---------------------------------------- 0.0/797.5 kB ? eta -:--:--
   ---------------------------------------- 797.5/797.5 kB 24.2 MB/s  0:00:00
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   --------------------- ------------------ 6.0/11.3 MB 29.6 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 28.8 MB/s  0:00:00
   ---------------------------------------- 0.0/28.0 MB ? eta -:--:--
   --------- ------------------------------ 6.8/28.0 MB 34.6 MB/s eta 0:00:01
   ------------------ --------------------- 13.1/28.0 MB 31.9 MB/s eta 0:00:01
   ------------------------------ --------- 21.2/28.0 MB 34.7 MB/s eta 0:00:01
   ---------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
!python -m pip install plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 7.2 MB/s eta 0:00:02
   --------------------------- ------------ 6.8/9.9 MB 24.6 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 22.8 MB/s  0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
!python -m streamlit run Home.py

^C
